# NFL ATS model playground
Edit the cells below and rerun them (Shift+Enter) to try new features or models.
**Cell order:** run 1 → 2 once, then rerun 3 → 5 as many times as you like.

In [ ]:
# 1. Setup and data load (cached in data/, so it's fast after the first run)
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from data import FIRST_SEASON, current_season, load_schedules, load_team_games
from features import FEATURES, build_dataset
from backtest import walk_forward, report, plot, upcoming_picks, logistic, gboost

sched = load_schedules()
team_games = load_team_games(range(FIRST_SEASON, current_season() + 1))
games = build_dataset(sched[sched.season >= FIRST_SEASON], team_games)
print(games.shape)
games.tail()

In [ ]:
# 2. Explore: how does each feature correlate with the home team covering?
played = games[games.home_cover.notna()]
played[FEATURES + ["home_cover"]].corr()["home_cover"].drop("home_cover").sort_values()

## 3. Add your own features
Create new columns on `games` here. They must only use information known **before kickoff**.

In [ ]:
games["home_dog"] = (games.spread_line < 0).astype(int)        # home team is the underdog
games["big_spread"] = (games.spread_line.abs() >= 7).astype(int)  # spread of a touchdown or more

MY_FEATURES = ["spread_line", "net_epa_diff", "rest_diff", "home_dog", "big_spread", "div_game"]
print("All available columns:", sorted(games.columns))

## 4. Define models
Each entry is `name: (feature list, function that returns an untrained sklearn model)`.
Keep `spread_only` as the control: it should land near 50%.

In [ ]:
models = {
    "spread_only": (["spread_line"], logistic),
    "my_logistic": (MY_FEATURES, lambda: make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.05, max_iter=1000))),
    "my_forest": (MY_FEATURES, lambda: make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=100, n_jobs=-1, random_state=0))),
}

In [ ]:
# 5. Walk-forward backtest: for each season, train only on earlier seasons
preds = walk_forward(games, first_test=2012, models=models)
summary = report(preds, models)
plot(preds, models);

## 6. Probabilities for upcoming games
These are the probabilities that the **home** team covers. Anything within about ±0.03 of 0.5 is effectively a coin flip.

In [ ]:
upcoming_picks(games, models)